<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install deepeval chromadb langchain langchain_community langchain_text_splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires openteleme

In [27]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [5]:
from deepeval.synthesizer import Synthesizer

synthesizer = Synthesizer()
goldens = synthesizer.generate_goldens_from_docs(
    document_paths=['sentiment.txt'], # Replace with your file
    include_expected_output=True
)
print(goldens)

Output()

[Confident AI Synthesizer Log] WARNING: Filtering not applied: Not enough chunks in smallest document

[Confident AI Synthesizer Log] SUCCESS: Successfully deleted: /tmp/deepeval_chroma_ph2yyx54

[Confident AI Synthesizer Log] SUCCESS: Context Construction: Utilizing 1 out of 1 chunks.

[Golden(input='How are positive, negative, and neutral sentiments distributed across these 20 customer reviews?', actual_output=None, expected_output='There are 7 positive, 7 negative, and 6 neutral sentiments among the 20 customer reviews.', context=["input: I absolutely love this product. It exceeded my expectations.\nlabel: positive\n\ninput: This is the worst experience I've ever had.\nlabel: negative\n\ninput: The service was okay, nothing special.\nlabel: neutral\n\ninput: I'm extremely happy with the results!\nlabel: positive\n\ninput: The package arrived late and damaged.\nlabel: negative\n\ninput: It works as intended.\nlabel: neutral\n\ninput: Fantastic quality and fast shipping!\nlabel: positive\n\ninput: I'm very disappointed with the customer support.\nlabel: negative\n\ninput: The product is fine for the price.\nlabel: neutral\n\ninput: This app makes my life so much easier.\nlabel: positive\n\ninput: I regret buying this.\nlabel: negative\n\ninput: The update didn’t chan

In [9]:
dataframe = synthesizer.to_pandas()
print(dataframe)

                                               input  \
0  How are positive, negative, and neutral sentim...   
1  Which specific feedback labels—positive, negat...   

                                       actual_output  \
0  To analyze the distribution of positive, negat...   
1  To assign feedback labels to user statements, ...   

                                     expected_output  \
0  There are 7 positive, 7 negative, and 6 neutra...   
1  Here are the feedback labels assigned to each ...   

                                             context retrieval_context  \
0  [input: I absolutely love this product. It exc...              None   
1  [input: I absolutely love this product. It exc...              None   

   n_chunks_per_context  context_length      evolutions context_quality  \
0                     1            1213  [Concretizing]            None   
1                     1            1213  [Concretizing]            None   

   synthetic_input_quality    source_file  


In [29]:
#getting actual output from llm around the golden dataset
# from openai import OpenAI
# client = OpenAI()

# for g in goldens:
#   response = client.chat.completions.create(
#       model="gpt-4o-mini",
#       messages=[{"role": "user", "content": g.input}]
#   )
#   g.actual_output = response.choices[0].message.content

from openai import OpenAI
client=OpenAI()

for g in goldens:
  prompt = f"""
You are given labeled customer reviews below.

Context:
{g.context}

Answer the following question strictly using the context.
Return only the final answer. Do not explain.

Question:
{g.input}
"""

  response = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=[{"role": "user", "content": prompt}] # Changed 'context' to 'content'
  )

  g.actual_output = response.choices[0].message.content.strip()

In [30]:
df = synthesizer.to_pandas()
df.head()

,input,actual_output,expected_output,context,retrieval_context,n_chunks_per_context,context_length,evolutions,context_quality,synthetic_input_quality,source_file
0,"How are positive, negative, and neutral sentim...","Positive: 8, Negative: 6, Neutral: 6","There are 7 positive, 7 negative, and 6 neutra...",[input: I absolutely love this product. It exc...,None,1,1213,[Concretizing],None,0.0,sentiment.txt
1,"Which specific feedback labels—positive, negat...","positive, negative, neutral, positive, negativ...",Here are the feedback labels assigned to each ...,[input: I absolutely love this product. It exc...,None,1,1213,[Concretizing],None,0.0,sentiment.txt


In [31]:
#converting to test cases
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=g.input,
        actual_output=g.actual_output,
        expected_output=g.expected_output,
        context=g.context
    )
    for g in goldens
]

In [32]:
#metric

from deepeval.metrics import GEval
from deepeval import evaluate
from deepeval.test_case import LLMTestCaseParams

correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output matches the expected output exactly for sentiment classification (positive, negative, neutral)",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ]
)

evaluate(test_cases=test_cases, metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases




Metrics Summary

  - ❌ Correctness [GEval] (score: 0.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The actual output does not match the expected output exactly: the counts for positive and negative sentiments are swapped (8 positive and 6 negative in actual vs. 7 positive and 7 negative in expected), while neutral is correct. According to the evaluation steps, any mismatch is marked as incorrect., error: None)

For test case:

  - input: How are positive, negative, and neutral sentiments distributed across these 20 customer reviews?
  - actual output: Positive: 8, Negative: 6, Neutral: 6
  - expected output: There are 7 positive, 7 negative, and 6 neutral sentiments among the 20 customer reviews.
  - context: ["input: I absolutely love this product. It exceeded my expectations.\nlabel: positive\n\ninput: This is the worst experience I've ever had.\nlabel: negative\n\ninput: The service was okay, nothing special.\nlabel: neutral\n\ninput: I'm extremely happy wit

⚠ WARNING: No hyperparameters logged.
» ]8;id=500480;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.11s | token cost: 0.005016 USD)
» Test Results (2 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=False, score=0.0, reason='The actual output does not match the expected output exactly: the counts for positive and negative sentiments are swapped (8 positive and 6 negative in actual vs. 7 positive and 7 negative in expected), while neutral is correct. According to the evaluation steps, any mismatch is marked as incorrect.', strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.0022719999999999997, verbose_logs='Criteria:\nDetermine whether the actual output matches the expected output exactly for sentiment classification (positive, negative, neutral) \n \nEvaluation Steps:\n[\n    "Review the Input to understand the context and content for sentiment classification.",\n    "Compare the Actual Output sentiment label to the Expected Output sentiment label.",\n    "Check if the Actual Output matches the Expected O

In [33]:
from deepeval.test_case import LLMTestCase

reviews = [
    ("I absolutely love this product. It exceeded my expectations.", "positive"),
    ("This is the worst experience I've ever had.", "negative"),
    ("The service was okay, nothing special.", "neutral"),
    ("I'm extremely happy with the results!", "positive"),
    ("The package arrived late and damaged.", "negative"),
    ("It works as intended.", "neutral"),
    ("Fantastic quality and fast shipping!", "positive"),
    ("I'm very disappointed with the customer support.", "negative"),
    ("The product is fine for the price.", "neutral"),
    ("This app makes my life so much easier.", "positive"),
    ("I regret buying this.", "negative"),
    ("The update didn’t change much.", "neutral"),
    ("Absolutely amazing experience from start to finish.", "positive"),
    ("It stopped working after two days.", "negative"),
    ("The design is simple and clean.", "neutral"),
    ("Best purchase I've made this year!", "positive"),
    ("Total waste of money.", "negative"),
    ("It’s neither good nor bad.", "neutral"),
    ("Customer service was incredibly helpful.", "positive"),
    ("I'm frustrated with the constant crashes.", "negative"),
]

test_cases = [
    LLMTestCase(
        input=text,
        actual_output="",
        expected_output=label
    )
    for text, label in reviews
]

In [34]:
from openai import OpenAI
client = OpenAI()

for test_case in test_cases:
    prompt = f"""
Classify the sentiment of the following text.

Return ONLY one word:
positive
negative
neutral

Text:
{test_case.input}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    test_case.actual_output = response.choices[0].message.content.strip().lower()

In [36]:
from deepeval.metrics import GEval
from deepeval import evaluate

classification_metric = GEval(
    name="Sentiment Classification Accuracy",
    criteria="Determine whether the predicted sentiment exactly matches the expected sentiment label.",
    evaluation_params=["actual_output", "expected_output"]
)

evaluate(test_cases=test_cases, metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4.1, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute:in _a_exe



Metrics Summary

  - ✅ Correctness [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The Actual Output sentiment label is 'positive', which matches the Expected Output exactly. The input expresses a clearly positive sentiment, and the response aligns perfectly with the evaluation steps., error: None)

For test case:

  - input: This app makes my life so much easier.
  - actual output: positive
  - expected output: positive
  - context: None
  - retrieval context: None


Metrics Summary

  - ✅ Correctness [GEval] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The Actual Output sentiment label 'negative' matches the Expected Output exactly for the clearly negative input 'Total waste of money.' This demonstrates full alignment with the evaluation steps., error: None)

For test case:

  - input: Total waste of money.
  - actual output: negative
  - expected output: negative
  - context: None
  - retrieval context: None


⚠ WARNING: No hyperparameters logged.
» ]8;id=12310;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 8.68s | token cost: 0.038474 USD)
» Test Results (20 total tests):
   » Pass Rate: 90.0% | Passed: 18 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_9', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=1.0, reason="The Actual Output sentiment label is 'positive', which matches the Expected Output exactly. The input expresses a clearly positive sentiment, and the response aligns perfectly with the evaluation steps.", strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.001948, verbose_logs='Criteria:\nDetermine whether the actual output matches the expected output exactly for sentiment classification (positive, negative, neutral) \n \nEvaluation Steps:\n[\n    "Read the Input to understand the context and sentiment being expressed.",\n    "Compare the Actual Output sentiment label to the Expected Output sentiment label.",\n    "Check if the Actual Output matches the Expected Output exactly (positive, negative, or neutral).",\n    "Mark as correct only if the Actual Output and Expected Output are iden